# CS60075 — NLP Assignment II

**Language Modelling on WikiText-2**

**Author : Akash Mandal**

**Roll No. : 24CS10001**

| Task | Description | Model |
|------|-------------|-------|
| 1 | N-gram Language Models with Smoothing | Add-k, Good-Turing, Kneser-Ney |
| 2 | Neural Language Model | LSTM |
| 3 | Hybrid Interpolation | KN + LSTM |

---
> **Run Instructions:** Use `Runtime → Run All` for a complete, zero-intervention execution.  
> **GPU:** Enable GPU via `Runtime → Change runtime type → T4 GPU` before running.

---
## 0. Setup — Dependencies & Reproducibility Seeds

In [1]:
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 datasets==3.1.0 nltk==3.9.1

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import math
import random
import warnings
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
from datasets import load_dataset

import torch
import torch.nn as nn
import torch.nn.functional as fnn
import numpy as np

warnings.filterwarnings("ignore")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All imports successful.")

c:\Users\makas\Documents\Code\NLP_Asgn2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Random seed: {SEED}")

Device: cuda
Random seed: 42


In [4]:
UNK_TOKEN  = "<UNK>"
BOS_TOKEN  = "<s>"
MAX_VOCAB  = 30000
MIN_FREQ   = 2
MAX_ORDER  = 4
GT_THRESH  = 5
KN_D       = 0.75

EMBED_DIM       = 400
HIDDEN_SIZE     = 400
NUM_LAYERS      = 3
BATCH_SIZE      = 80
EVAL_BATCH_SIZE = 3
BPTT            = 70
CLIP            = 0.25
LR              = 30.0
EPOCHS          = 50
WDROP           = 0.5
ODROP           = 0.4
EDROP           = 0.1

SAVE_PATH = "task2_best.pt"

print("Constants set.")

Constants set.


In [5]:
def load_and_tokenize_splits():
    print("Loading Salesforce/wikitext (wikitext-2-raw-v1) …")
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

    result = {}
    for name in ("train", "validation", "test"):
        print(f"  Tokenizing {name} split...")
        split_tokens = []
        for line in ds[name]["text"]:
            split_tokens.extend(word_tokenize(line.lower()))
        result[name] = split_tokens
        print(f"    → {len(result[name]):,} tokens")

    return result["train"], result["validation"], result["test"]


def build_and_apply_vocab(train_tokens, val_tokens, test_tokens):
    counts = Counter(train_tokens)
    valid_tokens = {w: c for w, c in counts.items() if c >= MIN_FREQ}
    top_tokens = Counter(valid_tokens).most_common(MAX_VOCAB)
    vocab = set(w for w, _ in top_tokens)
    vocab.add(UNK_TOKEN)

    def apply_unk(tokens):
        return [t if t in vocab else UNK_TOKEN for t in tokens]

    train_unk = apply_unk(train_tokens)
    val_unk   = apply_unk(val_tokens)
    test_unk  = apply_unk(test_tokens)

    print(f"Vocabulary size: {len(vocab):,}")
    return train_unk, val_unk, test_unk, vocab


train_tokens, val_tokens, test_tokens = load_and_tokenize_splits()
train_unk, val_unk, test_unk, vocab = build_and_apply_vocab(
    train_tokens, val_tokens, test_tokens
)
vocab_size = len(vocab)
print(f"\nData ready.  Vocab = {vocab_size:,} types.")

Loading Salesforce/wikitext (wikitext-2-raw-v1) …


  Tokenizing train split...
    → 2,098,704 tokens
  Tokenizing validation split...
    → 219,053 tokens
  Tokenizing test split...
    → 247,439 tokens
Vocabulary size: 30,001

Data ready.  Vocab = 30,001 types.


---
---
# Task 1 — N-gram Language Models with Smoothing

**Models:** Add-k (k = 1, 0.5, 0.1), Good-Turing, Kneser-Ney (4-gram)

**Metric:** Perplexity on WikiText-2 validation & test splits

In [6]:
class nGrams:

    def __init__(self, tokens: list, n: int = MAX_ORDER):
        self.n = n
        self.bos_token = BOS_TOKEN

        padding = [BOS_TOKEN] * (self.n - 1)
        self.tokens = padding + tokens

        self.ngrams = {}
        self.context_ngrams = {}
        self.counts = {}
        self.context_counts = {}

        for order in range(1, self.n + 1):
            self.ngrams[order] = self._generate(order)
            self.counts[order] = Counter(self.ngrams[order])

            if order == 1:
                self.context_ngrams[order] = self._generate(0)
                self.context_counts[order] = Counter(self.context_ngrams[order])
            else:
                self.context_ngrams[order] = self.ngrams[order - 1]
                self.context_counts[order] = self.counts[order - 1]

    def _generate(self, size: int):
        if size == 0:
            return [()] * len(self.tokens)
        return [tuple(self.tokens[i:i + size]) for i in range(len(self.tokens) - size + 1)]


print("Building N-Gram counts from training data (up to 4-grams)…")
ngrams_data = nGrams(train_unk)
print("Done.")

Building N-Gram counts from training data (up to 4-grams)…
Done.


In [13]:
class addKModel:

    def __init__(self, ngrams, k: float, vocab_size: int, n: int = None, backoff_alpha: float = 0.4):
        self.k = k
        self.ngrams = ngrams
        self.vocab_size = vocab_size
        self.n = n if n is not None else ngrams.n
        self.backoff_alpha = backoff_alpha

    def log_prob_token(self, token, context):
        target_len = self.n - 1
        if len(context) < target_len:
            context = (self.ngrams.bos_token,) * (target_len - len(context)) + context
        elif len(context) > target_len:
            context = context[-target_len:] if target_len > 0 else ()

        for current_n in range(self.n, 0, -1):
            curr_ctx = context[-(current_n - 1):] if current_n > 1 else ()
            count_ctx = self.ngrams.context_counts[current_n].get(curr_ctx, 0)
            
            if count_ctx > 0 or current_n == 1:
                count_ng = self.ngrams.counts[current_n].get(curr_ctx + (token,), 0)
                prob = (count_ng + self.k) / (count_ctx + self.k * self.vocab_size)
                backoff_steps = self.n - current_n
                penalty = backoff_steps * math.log(self.backoff_alpha)
                
                return math.log(prob) + penalty

        return math.log(1.0 / self.vocab_size)

    def log_prob_seq(self, seq):
        lp = 0.0
        for i in range(len(seq)):
            start = max(0, i - self.n + 1)
            lp += self.log_prob_token(seq[i], tuple(seq[start:i]))
        return lp
class goodTuringModel:

    def __init__(self, ngrams, vocab_size: int, threshold: int = GT_THRESH, backoff_alpha: float = 0.4):
        self.ngrams = ngrams
        self.threshold = threshold
        self.vocab_size = vocab_size
        self.backoff_alpha = backoff_alpha

        self.Nr = {}
        self.adjusted = {}
        for n in range(1, ngrams.n + 1):
            counts = self.ngrams.counts[n]
            self.Nr[n] = Counter(counts.values())

            N0 = max(0, (self.vocab_size ** n) - len(counts))
            self.Nr[n][0] = N0

            self.adjusted[n] = {}
            if N0 > 0 and self.Nr[n][1] > 0:
                self.adjusted[n][0] = self.Nr[n][1] / N0
            else:
                self.adjusted[n][0] = 1e-10

            for r in range(1, self.threshold + 1):
                if self.Nr[n][r] > 0 and self.Nr[n][r + 1] > 0:
                    self.adjusted[n][r] = (r + 1) * (self.Nr[n][r + 1] / self.Nr[n][r])
                else:
                    self.adjusted[n][r] = float(r)

    def log_prob_token(self, token, context):
        target_len = self.ngrams.n - 1
        if len(context) < target_len:
            context = (self.ngrams.bos_token,) * (target_len - len(context)) + context
        elif len(context) > target_len:
            context = context[-target_len:] if target_len > 0 else ()

        for n in range(self.ngrams.n, 0, -1):
            curr_ctx = context[-(n - 1):] if n > 1 else ()
            count_ctx = self.ngrams.context_counts[n].get(curr_ctx, 0)
            if count_ctx > 0:
                count_ng = self.ngrams.counts[n].get(curr_ctx + (token,), 0)
                if count_ng > 0 or n == 1:
                    c_star = self.adjusted[n].get(count_ng, float(count_ng))
                    
                    backoff_steps = self.ngrams.n - n
                    penalty = backoff_steps * math.log(self.backoff_alpha)
                    
                    return math.log(c_star / count_ctx) + penalty

        return math.log(1.0 / self.vocab_size)

    def log_prob_seq(self, seq):
        lp = 0.0
        for i in range(len(seq)):
            start = max(0, i - self.ngrams.n + 1)
            lp += self.log_prob_token(seq[i], tuple(seq[start:i]))
        return lp


class kneserNeyModel:

    def __init__(self, ngrams, vocab_size: int, d: float = KN_D, lam: float = 0.5):
        self.ngrams = ngrams
        self.vocab_size = vocab_size
        self.d = d
        self.lam = lam

        self._cont_uni = Counter()
        if 2 in self.ngrams.counts:
            for (w1, w2) in self.ngrams.counts[2]:
                self._cont_uni[w2] += 1
        self._cont_uni_total = sum(self._cont_uni.values())

        self._cont_bi = Counter()
        if 3 in self.ngrams.counts:
            for (w0, w1, w2) in self.ngrams.counts[3]:
                self._cont_bi[(w1, w2)] += 1
        self._cont_bi_ctx = Counter()
        for (w1, w2), cnt in self._cont_bi.items():
            self._cont_bi_ctx[(w1,)] += cnt

        self._cont_tri = Counter()
        if 4 in self.ngrams.counts:
            for (w0, w1, w2, w3) in self.ngrams.counts[4]:
                self._cont_tri[(w1, w2, w3)] += 1
        self._cont_tri_ctx = Counter()
        for (w1, w2, w3), cnt in self._cont_tri.items():
            self._cont_tri_ctx[(w1, w2)] += cnt

    def prob_token(self, token, context):
        target_len = self.ngrams.n - 1
        if len(context) < target_len:
            context = (self.ngrams.bos_token,) * (target_len - len(context)) + context
        elif len(context) > target_len:
            context = context[-target_len:] if target_len > 0 else ()
        return max(self._prob(token, context, is_highest=True), 1e-30)

    def log_prob_token(self, token, context):
        return math.log(self.prob_token(token, context))

    def _prob(self, token, context, is_highest=False):
        if len(context) == 0:
            cc = self._cont_uni.get(token, 0)
            if self._cont_uni_total > 0 and cc > 0:
                return cc / self._cont_uni_total
            return 1.0 / self.vocab_size

        if is_highest:
            order = len(context) + 1
            c_ng  = self.ngrams.counts[order].get(context + (token,), 0)
            c_ctx = self.ngrams.context_counts[order].get(context, 0)
        else:
            ctx_len = len(context)
            if ctx_len == 1:
                c_ng  = self._cont_bi.get(context + (token,), 0)
                c_ctx = self._cont_bi_ctx.get(context, 0)
            elif ctx_len == 2:
                c_ng  = self._cont_tri.get(context + (token,), 0)
                c_ctx = self._cont_tri_ctx.get(context, 0)
            else:
                order = len(context) + 1
                c_ng  = self.ngrams.counts[order].get(context + (token,), 0)
                c_ctx = self.ngrams.context_counts[order].get(context, 0)

        if c_ctx == 0:
            return self._prob(token, context[1:], is_highest=False)

        first_term = max(c_ng - self.d, 0.0) / c_ctx
        lower = self._prob(token, context[1:], is_highest=False)
        return first_term + self.lam * lower

    def log_prob_seq(self, seq):
        lp = 0.0
        for i in range(len(seq)):
            start = max(0, i - self.ngrams.n + 1)
            lp += self.log_prob_token(seq[i], tuple(seq[start:i]))
        return lp

    def perplexity(self, seq):
        return math.exp(-self.log_prob_seq(seq) / len(seq))

    def tune_lambda(self, val_tokens, grid=None):
        if grid is None:
            grid = [round(0.1 * i, 1) for i in range(1, 10)]
        best_lam, best_ppl = self.lam, float("inf")
        for lam in grid:
            self.lam = lam
            ppl = self.perplexity(val_tokens)
            print(f"  λ = {lam:.1f} → Val PPL = {ppl:.4f}")
            if ppl < best_ppl:
                best_ppl, best_lam = ppl, lam
        self.lam = best_lam
        print(f"  ✓ Best λ = {self.lam:.1f}  (Val PPL = {best_ppl:.4f})")
        return best_lam, best_ppl


def calculate_perplexity(model, tokens):
    if hasattr(model, 'perplexity'):
        return model.perplexity(tokens)
    lp = model.log_prob_seq(tokens)
    return math.exp(-lp / len(tokens))


print("Smoothing model classes defined.")

Smoothing model classes defined.


In [14]:
models_t1 = {
    "Add-1":       addKModel(ngrams_data, k=1.0,  vocab_size=vocab_size),
    "Add-0.5":     addKModel(ngrams_data, k=0.5,  vocab_size=vocab_size),
    "Add-0.1":     addKModel(ngrams_data, k=0.1,  vocab_size=vocab_size),
    "Good-Turing": goodTuringModel(ngrams_data, vocab_size=vocab_size),
    "Kneser-Ney":  kneserNeyModel(ngrams_data, vocab_size=vocab_size),
}

val_ppls = {}
print("Validation Perplexities")
print("=" * 50)
for name, model in models_t1.items():
    if name == "Kneser-Ney":
        print(f"\n  Tuning Kneser-Ney λ on validation set:")
        _, best_ppl = model.tune_lambda(val_unk)
        val_ppls[name] = best_ppl
    else:
        val_ppls[name] = calculate_perplexity(model, val_unk)
    print(f"  {name:<15s}  Val PPL = {val_ppls[name]:>12,.4f}")

best_model_name = min(val_ppls, key=val_ppls.get)
print(f"\n✓ Best model (lowest val PPL): {best_model_name}")

test_ppls = {}
for name, model in models_t1.items():
    test_ppls[name] = calculate_perplexity(model, test_unk)

kn_model = models_t1["Kneser-Ney"]

print("\n")
print("=" * 60)
print("  TASK 1 RESULTS — N-GRAM PERPLEXITY (WikiText-2)")
print("=" * 60)
print(f"  {'Model':<20s} | {'Val PPL':>12s} | {'Test PPL':>12s}")
print("-" * 60)
for name in models_t1:
    print(f"  {name:<20s} | {val_ppls[name]:>12,.2f} | {test_ppls[name]:>12,.2f}")
print("=" * 60)

Validation Perplexities
  Add-1            Val PPL =  22,344.0027
  Add-0.5          Val PPL =  17,748.6293
  Add-0.1          Val PPL =  10,110.8201
  Good-Turing      Val PPL =     872.0565

  Tuning Kneser-Ney λ on validation set:
  λ = 0.1 → Val PPL = 1120.0887
  λ = 0.2 → Val PPL = 594.0105
  λ = 0.3 → Val PPL = 402.8298
  λ = 0.4 → Val PPL = 302.5454
  λ = 0.5 → Val PPL = 240.4784
  λ = 0.6 → Val PPL = 198.2124
  λ = 0.7 → Val PPL = 167.5736
  λ = 0.8 → Val PPL = 144.3609
  λ = 0.9 → Val PPL = 126.1867
  ✓ Best λ = 0.9  (Val PPL = 126.1867)
  Kneser-Ney       Val PPL =     126.1867

✓ Best model (lowest val PPL): Kneser-Ney


  TASK 1 RESULTS — N-GRAM PERPLEXITY (WikiText-2)
  Model                |      Val PPL |     Test PPL
------------------------------------------------------------
  Add-1                |    22,344.00 |    21,444.11
  Add-0.5              |    17,748.63 |    16,979.46
  Add-0.1              |    10,110.82 |     9,614.14
  Good-Turing          |       872.06

---
---
# Task 2 — Neural Language Model (LSTM)

**Architecture:** 3-layer LSTM, embed=400, hidden=400, weight-tied  
**Regularisation:** DropConnect (0.5), Output Dropout (0.4), Embedding Dropout (0.1)  
**Optimiser:** SGD, lr=30, gradient clipping=0.25  
**Metric:** Test perplexity

In [9]:
class lockedDropout(nn.Module):
    def forward(self, x, dropout=0.5):
        if not self.training or dropout == 0.0:
            return x
        mask = x.new_empty(1, x.size(1), x.size(2)).bernoulli_(1 - dropout).div(1 - dropout)
        return x * mask


class lstm(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers,
                 dropout=ODROP, dropconnect=WDROP, embeddrop=EDROP,
                 tie_weights=True):
        super().__init__()
        self.vocab_size  = vocab_size
        self.embed_dim   = embed_dim
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.dropout     = dropout
        self.dropconnect = dropconnect
        self.embeddrop   = embeddrop
        self.tie_weights = tie_weights
        self.odrop       = dropout

        self.encoder  = nn.Embedding(vocab_size, embed_dim)
        self.lockdrop = lockedDropout()

        self.rnns = nn.ModuleList()
        for i in range(num_layers):
            inp = embed_dim if i == 0 else hidden_size
            self.rnns.append(nn.LSTM(input_size=inp, hidden_size=hidden_size, num_layers=1))

        self.decoder = nn.Linear(hidden_size, vocab_size)
        if tie_weights:
            self.decoder.weight = self.encoder.weight

        self._setup_dropconnect()
        self._init_weights()

    def _setup_dropconnect(self):
        for rnn in self.rnns:
            w = rnn.weight_hh_l0
            del rnn._parameters["weight_hh_l0"]
            rnn.register_parameter("weight_hh_raw", nn.Parameter(w.data))

    def _apply_dropconnect(self):
        for rnn in self.rnns:
            raw_w = getattr(rnn, "weight_hh_raw")
            if self.training and self.dropconnect > 0.0:
                w = fnn.dropout(raw_w, p=self.dropconnect, training=True)
            else:
                w = raw_w.clone()
            setattr(rnn, "weight_hh_l0", w)

    def _init_weights(self):
        nn.init.uniform_(self.encoder.weight, -0.1, 0.1)
        self.decoder.bias.data.zero_()

    def forward(self, input, hidden):
        if self.training and self.embeddrop > 0.0:
            mask = (
                self.encoder.weight
                .new_empty(self.vocab_size, 1)
                .bernoulli_(1 - self.embeddrop)
                .div(1 - self.embeddrop)
            )
            emb = fnn.embedding(input, self.encoder.weight * mask)
        else:
            emb = self.encoder(input)

        self._apply_dropconnect()

        raw_output = emb
        new_h, new_c = [], []
        for i, rnn in enumerate(self.rnns):
            raw_output, (h_i, c_i) = rnn(
                raw_output, (hidden[0][i:i+1], hidden[1][i:i+1])
            )
            new_h.append(h_i)
            new_c.append(c_i)
            if i < self.num_layers - 1:
                raw_output = self.lockdrop(raw_output, self.odrop)

        output  = self.lockdrop(raw_output, self.odrop)
        decoded = self.decoder(output.view(-1, self.hidden_size))

        h_n = torch.cat(new_h, dim=0)
        c_n = torch.cat(new_c, dim=0)
        return decoded, (h_n, c_n), raw_output

    def init_hidden(self, batch_size):
        w = next(self.parameters())
        return (
            w.new_zeros(self.num_layers, batch_size, self.hidden_size),
            w.new_zeros(self.num_layers, batch_size, self.hidden_size),
        )


print("LSTM model class defined.")

LSTM model class defined.


In [10]:
def word_mappings(vocab):
    vocab_list = sorted(list(vocab))
    w2i = {word: i for i, word in enumerate(vocab_list)}
    i2w = {i: word for i, word in enumerate(vocab_list)}
    return w2i, i2w


def tokens_to_tensor(tokens, w2i):
    return torch.LongTensor([w2i.get(tok, w2i[UNK_TOKEN]) for tok in tokens])


def make_batch(data, batch_size, device):
    nbatch = len(data) // batch_size
    data = data.narrow(0, 0, nbatch * batch_size)
    data = data.view(batch_size, -1).t().contiguous()
    return data.to(device)


def get_batch(source, i, bptt):
    seq_len = min(bptt, source.size(0) - 1 - i)
    data   = source[i : i + seq_len]
    target = source[i + 1 : i + 1 + seq_len].reshape(-1)
    return data, target


def repackage_hidden(h):
    if isinstance(h, torch.Tensor):
        return h.detach()
    return tuple(repackage_hidden(v) for v in h)


w2i, i2w = word_mappings(vocab)

train_data = make_batch(tokens_to_tensor(train_unk, w2i), BATCH_SIZE,      DEVICE)
val_data   = make_batch(tokens_to_tensor(val_unk,   w2i), EVAL_BATCH_SIZE, DEVICE)
test_data  = make_batch(tokens_to_tensor(test_unk,  w2i), EVAL_BATCH_SIZE, DEVICE)

print(f"Train shape: {tuple(train_data.shape)}")
print(f"Val   shape: {tuple(val_data.shape)}")
print(f"Test  shape: {tuple(test_data.shape)}")

Train shape: (26233, 80)
Val   shape: (73017, 3)
Test  shape: (82479, 3)


In [11]:
def train_epoch(model, data, optimizer, criterion, clip, bptt):
    model.train()
    total_loss, n_tokens = 0.0, 0
    hidden = model.init_hidden(data.size(1))

    for i in range(0, data.size(0) - 1, bptt):
        inp, tgt = get_batch(data, i, bptt)
        if inp.size(0) == 0:
            continue
        hidden = repackage_hidden(hidden)
        optimizer.zero_grad()
        output, hidden, _ = model(inp, hidden)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item() * tgt.numel()
        n_tokens   += tgt.numel()

    return total_loss / n_tokens


def evaluate_loss(model, data, criterion, bptt):
    model.eval()
    total_loss, n_tokens = 0.0, 0
    hidden = model.init_hidden(data.size(1))

    with torch.no_grad():
        for i in range(0, data.size(0) - 1, bptt):
            inp, tgt = get_batch(data, i, bptt)
            if inp.size(0) == 0:
                continue
            output, hidden, _ = model(inp, hidden)
            hidden = repackage_hidden(hidden)
            loss = criterion(output, tgt)
            total_loss += loss.item() * tgt.numel()
            n_tokens   += tgt.numel()

    return total_loss / n_tokens


print("Training utilities defined.")

Training utilities defined.


In [12]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

lstm_model = lstm(
    vocab_size   = vocab_size,
    embed_dim    = EMBED_DIM,
    hidden_size  = HIDDEN_SIZE,
    num_layers   = NUM_LAYERS,
    dropout      = ODROP,
    dropconnect  = WDROP,
    embeddrop    = EDROP,
    tie_weights  = True,
).to(DEVICE)

n_params = sum(p.numel() for p in lstm_model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Training for {EPOCHS} epochs...\n")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(lstm_model.parameters(), lr=LR)

best_val_loss = float("inf")
val_losses = []
non_mono = 5 
trigger_asgd = False

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_epoch(lstm_model, train_data, optimizer, criterion, CLIP, BPTT)
    val_loss = evaluate_loss(lstm_model, val_data, criterion, BPTT)
    val_losses.append(val_loss)

    print(f"| Epoch {epoch:3d}/{EPOCHS} | "
          f"Train Loss {tr_loss:.3f} | "
          f"Val Loss {val_loss:.3f} | "
          f"Val PPL {math.exp(val_loss):>9,.2f}")
          
    if not trigger_asgd and len(val_losses) > non_mono and val_loss > min(val_losses[:-non_mono]):
        print("Switching to ASGD optimizer...")
        trigger_asgd = True
        optimizer = torch.optim.ASGD(lstm_model.parameters(), lr=LR, t0=0, lambd=0.)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(lstm_model.state_dict(), SAVE_PATH)

lstm_model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
print("\n✓ Loaded best checkpoint.")

Model parameters: 15,880,001
Training for 50 epochs...

| Epoch   1/50 | Train Loss 7.278 | Val Loss 6.313 | Val PPL    551.76
| Epoch   2/50 | Train Loss 6.349 | Val Loss 5.900 | Val PPL    364.95
| Epoch   3/50 | Train Loss 6.025 | Val Loss 5.631 | Val PPL    278.84
| Epoch   4/50 | Train Loss 5.818 | Val Loss 5.514 | Val PPL    248.22
| Epoch   5/50 | Train Loss 5.667 | Val Loss 5.360 | Val PPL    212.68
| Epoch   6/50 | Train Loss 5.531 | Val Loss 5.233 | Val PPL    187.43
| Epoch   7/50 | Train Loss 5.435 | Val Loss 5.191 | Val PPL    179.59
| Epoch   8/50 | Train Loss 5.350 | Val Loss 5.094 | Val PPL    163.03
| Epoch   9/50 | Train Loss 5.283 | Val Loss 5.049 | Val PPL    155.80
| Epoch  10/50 | Train Loss 5.221 | Val Loss 5.031 | Val PPL    153.10
| Epoch  11/50 | Train Loss 5.165 | Val Loss 4.973 | Val PPL    144.53
| Epoch  12/50 | Train Loss 5.120 | Val Loss 4.939 | Val PPL    139.67
| Epoch  13/50 | Train Loss 5.072 | Val Loss 4.895 | Val PPL    133.56
| Epoch  14/50 | Trai

In [16]:
test_loss_t2 = evaluate_loss(lstm_model, test_data, criterion, BPTT)
test_ppl_t2  = math.exp(test_loss_t2)

print("\n" + "=" * 55)
print("  TASK 2 RESULTS — LSTM TEST PERPLEXITY")
print("=" * 55)
print(f"  Perplexity:  {test_ppl_t2:>12,.2f}")
print("=" * 55)


  TASK 2 RESULTS — LSTM TEST PERPLEXITY
  Perplexity:         97.80


---
---
# Task 3 — Hybrid Language Model (Kneser-Ney + LSTM)

**Method:** Linear interpolation  
$P_{\text{hybrid}}(w | ctx) = \lambda \cdot P_{\text{KN}}(w | ctx) + (1 - \lambda) \cdot P_{\text{LSTM}}(w | ctx)$  

**Tuning:** Grid search for λ on the validation set  
**Metric:** Test perplexity for KN-only, LSTM-only, and hybrid models

In [17]:
class hybridModel:

    def __init__(self, kn_model, lstm_model, w2i, device=DEVICE, lam=0.5):
        self.kn_model  = kn_model
        self.lstm_model = lstm_model
        self.w2i       = w2i
        self.device    = device
        self.lam       = lam
        self.n         = kn_model.ngrams.n

    def get_kn_probs(self, seq):
        probs = []
        for i in range(len(seq)):
            start = max(0, i - self.n + 1)
            context = tuple(seq[start:i])
            probs.append(self.kn_model.prob_token(seq[i], context))
        return torch.tensor(probs, dtype=torch.float64)

    def get_lstm_probs(self, seq, bptt=BPTT):
        self.lstm_model.eval()
        bos_id  = self.w2i.get(BOS_TOKEN, self.w2i[UNK_TOKEN])
        seq_ids = [bos_id] + [self.w2i.get(tok, self.w2i[UNK_TOKEN]) for tok in seq]
        data    = torch.LongTensor(seq_ids).unsqueeze(1).to(self.device)

        hidden    = self.lstm_model.init_hidden(1)
        all_probs = []

        with torch.no_grad():
            for i in range(0, data.size(0) - 1, bptt):
                inp, tgt = get_batch(data, i, bptt)
                if inp.size(0) == 0:
                    continue
                output, hidden, _ = self.lstm_model(inp, hidden)
                hidden = repackage_hidden(hidden)
                probs  = fnn.softmax(output, dim=-1)
                tok_p  = probs.gather(1, tgt.unsqueeze(1)).squeeze(1)
                all_probs.append(tok_p.cpu().to(torch.float64))

        return torch.cat(all_probs, dim=0)

    def log_prob_from_probs(self, kn_probs, lstm_probs, lam=None):
        if lam is None:
            lam = self.lam
        hybrid = lam * kn_probs + (1.0 - lam) * lstm_probs
        hybrid = torch.clamp(hybrid, min=1e-30)
        return float(torch.sum(torch.log(hybrid)).item())

    def perplexity_from_probs(self, kn_probs, lstm_probs, lam=None):
        lp = self.log_prob_from_probs(kn_probs, lstm_probs, lam=lam)
        return math.exp(-lp / len(kn_probs))

    def tune_lambda(self, val_tokens, grid=None):
        if grid is None:
            grid = [round(0.1 * i, 1) for i in range(0, 11)]
        print("  Pre-computing validation probabilities...")
        kn_probs  = self.get_kn_probs(val_tokens)
        lstm_probs = self.get_lstm_probs(val_tokens)

        best_lam, best_ppl = self.lam, float("inf")
        for lam in grid:
            ppl = self.perplexity_from_probs(kn_probs, lstm_probs, lam=lam)
            print(f"    λ = {lam:.1f} → Val PPL = {ppl:.4f}")
            if ppl < best_ppl:
                best_ppl, best_lam = ppl, lam

        self.lam = best_lam
        print(f"  ✓ Best hybrid λ = {self.lam:.1f}  (Val PPL = {best_ppl:.4f})")
        return best_lam, best_ppl


print("Hybrid model class defined.")

Hybrid model class defined.


In [18]:
hybrid = hybridModel(kn_model=kn_model, lstm_model=lstm_model, w2i=w2i, device=DEVICE)

print("Tuning hybrid λ on validation set...")
best_lam, _ = hybrid.tune_lambda(val_unk)

print("\nEvaluating on test set...")
kn_test_probs  = hybrid.get_kn_probs(test_unk)
lstm_test_probs = hybrid.get_lstm_probs(test_unk)

kn_test_ppl     = hybrid.perplexity_from_probs(kn_test_probs, lstm_test_probs, lam=1.0)
lstm_test_ppl    = hybrid.perplexity_from_probs(kn_test_probs, lstm_test_probs, lam=0.0)
hybrid_test_ppl = hybrid.perplexity_from_probs(kn_test_probs, lstm_test_probs, lam=best_lam)

Tuning hybrid λ on validation set...
  Pre-computing validation probabilities...
    λ = 0.0 → Val PPL = 103.4113
    λ = 0.1 → Val PPL = 90.3579
    λ = 0.2 → Val PPL = 86.2798
    λ = 0.3 → Val PPL = 84.3939
    λ = 0.4 → Val PPL = 83.8410
    λ = 0.5 → Val PPL = 84.3668
    λ = 0.6 → Val PPL = 85.9695
    λ = 0.7 → Val PPL = 88.8667
    λ = 0.8 → Val PPL = 93.6461
    λ = 0.9 → Val PPL = 101.9706
    λ = 1.0 → Val PPL = 126.1867
  ✓ Best hybrid λ = 0.4  (Val PPL = 83.8410)

Evaluating on test set...


In [19]:
print("\n" + "=" * 60)
print("  TASK 3 RESULTS — FINAL COMPARISON TABLE (TEST PERPLEXITY)")
print("=" * 60)
print(f"  {'Model':<35s} | {'Test PPL':>15s}")
print("-" * 60)
print(f"  {'Task 1: Kneser-Ney (4-gram)':<35s} | {kn_test_ppl:>15,.2f}")
print(f"  {'Task 2: LSTM':<35s} | {lstm_test_ppl:>15,.2f}")
print(f"  {f'Task 3: Hybrid (λ={best_lam:.1f})':<35s} | {hybrid_test_ppl:>15,.2f}")
print("=" * 60)


  TASK 3 RESULTS — FINAL COMPARISON TABLE (TEST PERPLEXITY)
  Model                               |        Test PPL
------------------------------------------------------------
  Task 1: Kneser-Ney (4-gram)         |          118.63
  Task 2: LSTM                        |           97.78
  Task 3: Hybrid (λ=0.4)              |           80.06
